# Ajuste de Hiperparámetros - Validación Cruzada Temporal (TimeSeriesSplit)

Utilizaremos `RandomizedSearchCV` junto con `TimeSeriesSplit` para respetar el orden cronológico de los datos durante la búsqueda.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos

In [ ]:
base_path = '../data/model_input/cv/'

X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

print(f"X_train_full shape: {X_train_full.shape}")

### 2. Preprocesamiento y Configuración de Validación

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Random Forest - RandomizedSearchCV

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=tscv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

rf_random.fit(X_train_full, y_train_full)

print("Mejores hiperparámetros para Random Forest:")
print(rf_random.best_params_)
print(f"Mejor AUC promedio en CV: {rf_random.best_score_:.4f}")

### 4. Gradient Boosting - RandomizedSearchCV

In [ ]:
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

gb_random = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_grid,
    n_iter=15,
    cv=tscv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_random.fit(X_train_full, y_train_full)

print("Mejores hiperparámetros para Gradient Boosting:")
print(gb_random.best_params_)
print(f"Mejor AUC promedio en CV: {gb_random.best_score_:.4f}")

### 5. Comparación de Modelos

In [ ]:
# 1. Extraer el mejor score (AUC promedio en los folds de validación) y la desviación estándar
rf_best_score = rf_random.best_score_
rf_std = rf_random.cv_results_['std_test_score'][rf_random.best_index_]

gb_best_score = gb_random.best_score_
gb_std = gb_random.cv_results_['std_test_score'][gb_random.best_index_]

# 2. Crear un DataFrame para comparar
comparativa_cv = pd.DataFrame({
    'Modelo': ['Random Forest', 'Gradient Boosting'],
    'Mejor AUC Promedio (CV)': [rf_best_score, gb_best_score],
    'Desviación Estándar (Estabilidad)': [rf_std, gb_std]
})

# Mostrar la tabla ordenada por el mejor AUC
display(comparativa_cv.sort_values(by='Mejor AUC Promedio (CV)', ascending=False))

# 3. (Opcional) Graficar para verlo más visual
plt.figure(figsize=(8, 5))
sns.barplot(x='Modelo', y='Mejor AUC Promedio (CV)', data=comparativa_cv, palette='viridis')
plt.title('Comparativa de Modelos en Validación Cruzada (TimeSeriesSplit)')
plt.ylim(0.8, 1.0) # Ajusta el límite según tus resultados
plt.ylabel('AUC ROC Promedio')
plt.show()

### 6. Extraemos las métricas de los modelos con los hiperparámetros ajustados

In [ ]:
from sklearn.model_selection import cross_validate

# 1. Definimos todas las métricas que queremos extraer
metricas = {
    'auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# 2. Calculamos las métricas para el MEJOR Random Forest a través de los 5 folds
print("Calculando métricas en CV para Random Forest...")
cv_rf = cross_validate(rf_random.best_estimator_, X_train_full, y_train_full, cv=tscv, scoring=metricas)

# 3. Calculamos las métricas para el MEJOR Gradient Boosting a través de los 5 folds
print("Calculando métricas en CV para Gradient Boosting...")
cv_gb = cross_validate(gb_random.best_estimator_, X_train_full, y_train_full, cv=tscv, scoring=metricas)

# 4. Imprimimos los resultados promediados listos para tu tabla Markdown
print("\n" + "="*50)
print("RESULTADOS PROMEDIO EN VALIDACIÓN CRUZADA (5 Folds)")
print("="*50)

print("\n--- Random Forest (Tuned) ---")
print(f"ROC AUC:   {cv_rf['test_auc'].mean():.4f}")
print(f"Accuracy:  {cv_rf['test_accuracy'].mean():.4f}")
print(f"Precision: {cv_rf['test_precision'].mean():.4f}")
print(f"Recall:    {cv_rf['test_recall'].mean():.4f}")
print(f"F1-Score:  {cv_rf['test_f1'].mean():.4f}")

print("\n--- Gradient Boosting (Tuned) ---")
print(f"ROC AUC:   {cv_gb['test_auc'].mean():.4f}")
print(f"Accuracy:  {cv_gb['test_accuracy'].mean():.4f}")
print(f"Precision: {cv_gb['test_precision'].mean():.4f}")
print(f"Recall:    {cv_gb['test_recall'].mean():.4f}")
print(f"F1-Score:  {cv_gb['test_f1'].mean():.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extraer los resultados de la búsqueda de Random Forest
resultados_rf = pd.DataFrame(rf_random.cv_results_)

# Filtrar solo las columnas que nos interesan
columnas_interes = [
    'param_classifier__min_samples_leaf', 
    'param_classifier__max_depth', 
    'mean_test_score', 
    'std_test_score'
]
resultados_rf = resultados_rf[columnas_interes].copy()

# Limpiar los nombres para que sea más fácil trabajar
resultados_rf.rename(columns={
    'param_classifier__min_samples_leaf': 'min_samples_leaf',
    'param_classifier__max_depth': 'max_depth',
    'mean_test_score': 'mean_auc',
    'std_test_score': 'std_auc'
}, inplace=True)

# Crear el gráfico
fig, ax = plt.subplots(figsize=(10, 6))

# Agrupar por min_samples_leaf
for min_hoja, grupo in resultados_rf.groupby("min_samples_leaf"):
    grupo = grupo.copy()
    
    # Manejar el caso donde max_depth es None (Sin límite)
    # Le asignamos un valor numérico alto (ej. 50) solo para el gráfico
    profundidad_maxima_grafico = 50 
    grupo["profundidad_grafico"] = grupo["max_depth"].map(
        lambda valor: profundidad_maxima_grafico if pd.isna(valor) else int(valor)
    )
    
    # Ordenar por profundidad para que la línea se dibuje correctamente
    grupo = grupo.sort_values("profundidad_grafico")
    
    # Graficar con barras de error
    ax.errorbar(
        grupo["profundidad_grafico"],
        grupo["mean_auc"],
        yerr=grupo["std_auc"],
        marker="o",
        capsize=4,
        linewidth=2,
        label=f"min_samples_leaf={min_hoja}"
    )

# Configuración del gráfico
ax.set_title("Rendimiento de Random Forest en Validación Cruzada\n(AUC medio ± 1 desvío estándar)", fontsize=14, pad=15)
ax.set_xlabel("Profundidad Máxima del Árbol (max_depth)", fontsize=12)
ax.set_ylabel("ROC AUC Promedio", fontsize=12)

# Configurar las etiquetas del eje X
etiquetas_x = sorted([d for d in resultados_rf['max_depth'].unique() if pd.notna(d)])
posiciones_x = etiquetas_x + [profundidad_maxima_grafico]
nombres_x = [str(int(d)) for d in etiquetas_x] + ["Sin límite"]

ax.set_xticks(posiciones_x)
ax.set_xticklabels(nombres_x)

ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(title="Mínimo de muestras por hoja", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()